In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False



# 1. 경로 설정
BASE_DIR = Path(os.getcwd()).parent
PROCESSED_DIR = BASE_DIR / "01_data" / "02_processed"
MODEL_DIR = BASE_DIR / "03_models" # 모델 저장 폴더 (없으면 생성)
LABELED_PATH = PROCESSED_DIR / "03_sensor_labeled_data.csv"

if not MODEL_DIR.exists():
    MODEL_DIR.mkdir(parents=True)

# 2. 데이터 로드
df = pd.read_csv(LABELED_PATH)
print(f"📂 학습용 라벨링 데이터 로드 완료: {len(df):,}행")

# 3. 특징(X) 및 타겟(y) 설정
# 3단계 로직에 사용된 핵심 피처들을 선택합니다.
features = ['avg_corr', 'peak_std', 'critical_value', 'daily_peak', 'peak_ratio', 'spread']
X = df[features]
y = df['label']

# 4. 데이터 분할 (8:2)
# 4-1. 첫 번째 분할: 전체의 20%를 최종 테스트(Test)셋으로 분리
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4-2. 두 번째 분할: 남은 80% 중 12.5%를 검증(Val)셋으로 분리
# (전체 대비 0.8 * 0.125 = 0.1, 즉 전체의 10%가 Validation이 됨)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)
# 4-3 데이터 분할 결과 확인
print(f"📊 [데이터 분할 결과]")
print(f"- Train      : {len(X_train)}개 (70%)")
print(f"- Validation : {len(X_val)}개 (10%)")
print(f"- Test       : {len(X_test)}개 (20%)")

# 5. 모델 생성 및 훈련
# 데이터 불균형(6%의 위험군)을 해결하기 위해 class_weight='balanced'를 적용합니다.
print("\n🚀 Random Forest 모델 학습 시작...")
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12,
    min_samples_split=5,
    class_weight='balanced', # 불균형 데이터 처리 핵심 옵션
    random_state=42,
    n_jobs=-1 # 모든 CPU 코어 사용
)

rf_model.fit(X_train, y_train)

# 6. 모델 및 테스트 데이터 저장 (평가 시 사용)
joblib.dump(rf_model, MODEL_DIR / "leak_rf_model_v1.pkl")

# 평가용 데이터도 따로 저장해두면 06번 파일에서 불러오기 편합니다.
test_data = X_test.copy()
test_data['label'] = y_test
test_data.to_csv(PROCESSED_DIR / "04_model_test_data.csv", index=False)


📂 학습용 라벨링 데이터 로드 완료: 15,833행
📊 [데이터 분할 결과]
- Train      : 11082개 (70%)
- Validation : 1584개 (10%)
- Test       : 3167개 (20%)

🚀 Random Forest 모델 학습 시작...
